# Import Modules

In [1]:
import numpy as np
import pandas as pd 
import random

from collections import Counter
from sklearn.metrics import confusion_matrix

from sklearn.metrics import roc_auc_score, roc_curve, brier_score_loss                  
import matplotlib.pyplot as plt

import seaborn as sns
import tensorflow as tf                   

from tensorflow.keras.initializers import he_normal                                     
from sklearn.model_selection import train_test_split       

from tensorflow.keras import regularizers                                             
from joblib import dump, load               
                                            
from sklearn.preprocessing import StandardScaler                                        

2026-04-22 18:45:25.216398: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

# Choosing lead time

In [3]:
lead_time = 6

# Import and split data

In [4]:
test = pd.read_csv("/home/users/mendrika/NCAST/Data/Dakar/data-test-dakar.csv")
train_full = pd.read_csv("/home/users/mendrika/NCAST/Data/Dakar/data-train-dakar.csv")

In [5]:
val   = train_full[train_full["year"] == 2019].copy()
train = train_full[train_full["year"] != 2019].copy()

# Selecting Header

In [6]:
def choose_header(n, location, lead_time):
    """
    Generate header names for n closest storms.

    Returns:
        tuple of two lists:
            - input headers: year, month, day, hour, minute, lat1..n, lon1..n, wp1..n, size1..n, d1..n, mask1..n
            - target header: Cb_{location}_t{lead_time}
    """
    input_headers = ['year', 'month', 'day', 'hour', 'minute']
    
    for prefix in ['lat', 'lon', 'wp', 'size', 'd', 'mask']:
        input_headers.extend([f'{prefix}{i}' for i in range(1, n + 1)])
    
    target_header = f'Cb_{location}_t{lead_time}'
    
    return input_headers, target_header

In [7]:
input_headers, target_header = choose_header(3, "dakar", lead_time)

In [8]:
X_train = train[input_headers].copy()
X_val   = val[input_headers].copy()
X_test  = test[input_headers].copy()

y_train = train[target_header]
y_val   = val[target_header]
y_test  = test[target_header]

# Log transforming

In [9]:
cols_to_log = ['size1', 'size2', 'size3', 
               'wp1', 'wp2', 'wp3', 
               'd1', 'd2', 'd3']

for col in cols_to_log:
    if col in X_train.columns:
        X_train[col] = np.log1p(X_train[col])
        X_val[col]   = np.log1p(X_val[col])
        X_test[col]  = np.log1p(X_test[col])

# Scaling

In [10]:
mask_cols = ['mask1', 'mask2', 'mask3']

# Columns grouped per core
core_groups = {
    1: ['lat1', 'lon1', 'wp1', 'size1', 'd1'],
    2: ['lat2', 'lon2', 'wp2', 'size2', 'd2'],
    3: ['lat3', 'lon3', 'wp3', 'size3', 'd3']
}

# Copy datasets
X_train_scaled = X_train.copy()
X_val_scaled   = X_val.copy()
X_test_scaled  = X_test.copy()

scalers = {}

for i, cols in core_groups.items():
    mask = X_train[f'mask{i}'] == 1
    
    scaler = StandardScaler()
    scaler.fit(X_train.loc[mask, cols])  # fit only on real cores
    
    scalers[i] = scaler
    
    # Apply to full dataset (including masked rows)
    X_train_scaled.loc[:, cols] = scaler.transform(X_train[cols])
    X_val_scaled.loc[:, cols]   = scaler.transform(X_val[cols])
    X_test_scaled.loc[:, cols]  = scaler.transform(X_test[cols])

In [11]:
from joblib import dump

dump(scalers, f"scalers_t{lead_time}.pkl")

['scalers_t6.pkl']

# Model design

In [12]:
X_train_arr = X_train_scaled.to_numpy().astype("float32")
X_val_arr   = X_val_scaled.to_numpy().astype("float32")

y_train_arr = y_train.astype("float32").to_numpy()
y_val_arr   = y_val.astype("float32").to_numpy()

## MLP

In [13]:
best_auc = -1
best_config = None
best_model = None

for lr in [1e-3, 5e-4, 1e-4]:
    for dropout in [0.0, 0.2, 0.4]:
        for (h1, h2) in [(32, 8), (64, 16), (128, 32)]:

            model = tf.keras.models.Sequential([
                tf.keras.layers.Input(shape=(X_train_arr.shape[1],)),
                tf.keras.layers.Dense(h1, activation='relu', kernel_initializer=he_normal()),
                tf.keras.layers.Dropout(dropout),
                tf.keras.layers.Dense(h2, activation='relu'),
                tf.keras.layers.Dropout(dropout),
                tf.keras.layers.Dense(8, activation='relu'),
                tf.keras.layers.Dense(1, activation='sigmoid')
            ])

            model.compile(
                optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
                loss='binary_crossentropy',
                metrics=[tf.keras.metrics.AUC(name='auc')]
            )

            early_stopping = tf.keras.callbacks.EarlyStopping(
                monitor="val_auc",
                patience=5,
                mode='max',
                restore_best_weights=True
            )

            history = model.fit(
                X_train_arr,
                y_train_arr,
                validation_data=(X_val_arr, y_val_arr),
                epochs=50,
                batch_size=64,
                verbose=0,
                callbacks=[early_stopping]
            )

            val_auc = max(history.history["val_auc"])

            if val_auc > best_auc:
                best_auc = val_auc
                best_config = (lr, dropout, h1, h2)
                best_model = model  # save model

print("Best config:", best_config)
print("Best val ROC AUC:", best_auc)

Best config: (0.001, 0.2, 128, 32)
Best val ROC AUC: 0.8319482803344727


In [14]:
y_test_arr = y_test.astype("float32").to_numpy()
X_test_arr = X_test_scaled.to_numpy().astype("float32")

y_test_pred = best_model.predict(X_test_arr).ravel()

test_auc = roc_auc_score(y_test_arr, y_test_pred)
test_brier = brier_score_loss(y_test_arr, y_test_pred)

print("Test ROC AUC:", test_auc)
print("Test Brier score:", test_brier)

938/938 [==============================] - 1s 548us/step
Test ROC AUC: 0.7401972840524944
Test Brier score: 0.01145988792707088


In [15]:
best_model.save(f"best_mlp_model_t{lead_time}.keras")

## Logistic regression

In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

C_values = [0.01, 0.1, 1, 10]

best_auc = -1
best_model = None
best_C = None

for C in C_values:
    model = LogisticRegression(
        C=C,
        max_iter=2000,
        n_jobs=-1
    )
    
    model.fit(X_train_arr, y_train_arr)
    
    y_val_pred = model.predict_proba(X_val_arr)[:, 1]
    val_auc = roc_auc_score(y_val_arr, y_val_pred)
    
    if val_auc > best_auc:
        best_auc = val_auc
        best_model = model
        best_C = C

print("Best C:", best_C)
print("Best val ROC AUC:", best_auc)

Best C: 10
Best val ROC AUC: 0.8506193793795136


In [ ]:
# Predict probabilities
y_test_pred = best_model.predict_proba(X_test_arr)[:, 1]

# ROC AUC
test_auc = roc_auc_score(y_test_arr, y_test_pred)

# Brier score
test_brier = brier_score_loss(y_test_arr, y_test_pred)

print("Logistic Test ROC AUC:", test_auc)
print("Logistic Test Brier:", test_brier)

Logistic Test ROC AUC: 0.7436840279669953
Logistic Test Brier: 0.011462104872816486


In [18]:
best_model

LogisticRegression(C=10, max_iter=2000, n_jobs=-1)